In [28]:
%run 00_utils.ipynb

%load_ext autoreload
%autoreload 2

Root-Verzeichnis: /home/jessh/Study/Bachelor-Thesis
Datensätze: Anti-UAV410 + CST-AntiUAV
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
from pathlib import Path
import cv2, os
import numpy as np
from data_handler.dataset import AntiUAVDataset

import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider

In [30]:
# Trainings-Frames aller Datensätze in DATASETS (Reihenfolge wie in DATASETS).
# Nur einer: AntiUAVDataset(DatasetName.CST_ANTI_UAV, split="train", debug=True)
train_data = AntiUAVDataset(DATASETS, split="train", debug=True)

# Index-Bereich je Datensatz (für IMAGE_ID / train_data[...])
_start = 0
for ds in DATASETS:
    n = sum(r.dataset is ds for r in train_data.samples)
    print(f"{ds.value:<14} Index {_start} .. {_start + n - 1}")
    _start += n

24-11-2026 15:03:04 | INFO    | [train] Anti-UAV410: 213995 frames in 200 sequences (1.6 s)
24-11-2026 15:03:06 | INFO    | [train] CST-AntiUAV: 127991 frames in 120 sequences (1.2 s)


Anti-UAV410    Index 0 .. 213994
CST-AntiUAV    Index 213995 .. 341985


In [31]:
def visualise(img_org, img_aug):
    _, ax = plt.subplots(1, 3, figsize=(15, 5))
    vmin, vmax = img_org.min(), img_org.max()   # gleicher Skalenbereich für faire Vergleichbarkeit
    
    ax[0].imshow(img_org, cmap='gray', vmin=vmin, vmax=vmax); ax[0].set_title(f"original")
    ax[1].imshow(img_aug, cmap='gray', vmin=vmin, vmax=vmax); ax[1].set_title(f"augmented")
    ax[2].imshow(img_aug.astype(np.float32) - img_org.astype(np.float32), cmap='seismic');          ax[2].set_title("difference")
    
    for a in ax: 
        a.axis('off')
        
    plt.tight_layout()
    plt.show()

In [32]:
img, target, img_org = train_data[100000]

In [33]:
from augmentations.ir_camara_effects import netd_noise

# NETD-ähnliches Rauschen: T_noise = T_ref + eta, eta ~ N(0, sigma^2), sigma relativ zu [0, 1]
def explore(sigma=0.0159):
    out = netd_noise(img_org, sigma=sigma)
    visualise(img_org, out)

interact(
    explore,
    sigma = FloatSlider(min=0.0, max=0.05, step=0.0005, value=0.0, readout_format='.4f',
                        continuous_update=False, description='sigma'),
);

interactive(children=(FloatSlider(value=0.0, continuous_update=False, description='sigma', max=0.05, readout_f…

In [34]:
from augmentations.ir_camara_effects import blurring_with_psf

def explore(sigma_0=0.3, rho=1.0, theta_deg=0):
    out = blurring_with_psf(
        img_org, sigma_0=sigma_0, rho=rho,
        theta=np.radians(theta_deg)
    )
    visualise(img_org, out)

WIDE = {'description_width': 'initial'}   # volle Beschriftung anzeigen

interact(
    explore,
    sigma_0   = FloatSlider(min=0.1,  max=4.0, step=0.1,  value=0.1, continuous_update=False, style=WIDE,
                            description='blur strength'),
    rho       = FloatSlider(min=0.15, max=1.0, step=0.05, value=1.0, continuous_update=False, style=WIDE,
                            description='stretching'),
    theta_deg = FloatSlider(min=0,    max=180, step=5,    value=0,   continuous_update=False, style=WIDE,
                            description='direction'),
);

interactive(children=(FloatSlider(value=0.1, continuous_update=False, description='blur strength', max=4.0, mi…

In [35]:
from augmentations.ir_camara_effects import fpn

def explore(sigma_gain=0.01, sigma_offset=2.0, mode="pixel"):
    out = fpn(
        img_org, 
        sigma_gain=sigma_gain, 
        sigma_offset=sigma_offset, 
        mode=mode
    )
    
    visualise(img_org, out)

# Interaktiver Widget Explorer für das neue FPN
interact(
    explore,
    sigma_gain   = FloatSlider(min=0.0, max=0.5, step=0.025,  value=0.0, readout_format='.3f', continuous_update=False),
    sigma_offset = FloatSlider(min=0.0, max=10.0, step=0.5,   value=0.0, readout_format='.1f', continuous_update=False),
    mode         = widgets.Dropdown(options=["pixel", "column", "row"], value="pixel", description="mode")
);

interactive(children=(FloatSlider(value=0.0, continuous_update=False, description='sigma_gain', max=0.5, reado…

In [ ]:
from augmentations.ir_camara_effects import thermal_histogram_shift

def explore(delta_limit=30):
    out = thermal_histogram_shift(
        img_org, 
        delta_limit
    )

    visualise(img_org, out)

interact(
    explore,
    delta_limit   = FloatSlider(min=0.0, max=100,   step=5,   value=0.0, readout_format='.2f', continuous_update=False),
);

interactive(children=(FloatSlider(value=0.0, continuous_update=False, description='delta_limit', step=5.0), Ou…

In [ ]:
from augmentations.ir_camara_effects import salt_and_peper

def explore(p, q):
    out = salt_and_peper(
        img_org, 
        p=p, q=q
    )

    visualise(img_org, out)

WIDE = {'description_width': 'initial'}   # volle Beschriftung anzeigen

interact(
    explore,
    p = FloatSlider(min=0.0, max=0.12, step=0.005, value=0.0, readout_format='.3f', continuous_update=True, style=WIDE,
                    description='Black pixel'),
    q = FloatSlider(min=0.0, max=0.12, step=0.005, value=0.0, readout_format='.3f', continuous_update=True, style=WIDE,
                    description='White pixel'),
);

interactive(children=(FloatSlider(value=0.0, description='Black pixel', max=0.12, readout_format='.3f', step=0…